In [3]:
!pip install sentence-transformers pandas scikit-learn numpy

In [4]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances, manhattan_distances

c:\Users\ANANDHU\OneDrive\Desktop\Querytube AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
videos_df = pd.read_csv("cleaned_transcripts.csv")
queries_df = pd.read_csv("query_video_mapping.csv")

queries = queries_df["query"].tolist()
ground_truth = queries_df["relevant_video_id"].tolist()

videos_df.head()

,video_id,title,datetime,transcript
0,E-CH3-VyVck,Why is Git INSANELY Fast? (And How Commits Ac...,2026-02-24 11:00:29+00:00,You have typed get commit thousands of times. ...
1,URI5GsOBznk,YouTube Recommendation Engine: Complete Meltdo...,2026-02-21 04:26:16+00:00,"YouTube went down. [music] 350,000 users repor..."
2,8d2eG7bdepQ,Implementing OAuth and MFA: Full Authenticatio...,2026-02-18 11:00:16+00:00,Every time you click sign in with Google or co...
3,GQ6piqfwr5c,"How Stripe Built AI Agents That Write 1,000+ P...",2026-02-14 13:02:10+00:00,Scribe just revealed something big. They have ...
4,p5hA8rpCRXw,The Selenium Problem: Why QA Teams Waste 40% o...,2026-02-11 11:01:25+00:00,Here's a stat that surprised me. QA team spend...


In [6]:
models = [
    "all-MiniLM-L6-v2",
    "all-mpnet-base-v2",
    "multi-qa-MiniLM-L6-cos-v1"
]

In [7]:
all_results = []
ranking_output = []
evaluation_output = []

for model_name in models:

    print(f"\nEvaluating {model_name}")

    model = SentenceTransformer(model_name)

    video_texts = (videos_df["title"] + " " + videos_df["transcript"]).tolist()

    video_embeddings = model.encode(video_texts, show_progress_bar=True)
    query_embeddings = model.encode(queries, show_progress_bar=True)

    metrics = {
        "cosine": cosine_similarity,
        "euclidean": euclidean_distances,
        "manhattan": manhattan_distances
    }

    for metric_name, metric_func in metrics.items():

        ranks = []

        for i, q_emb in enumerate(query_embeddings):

            q_emb = q_emb.reshape(1, -1)

            scores = metric_func(q_emb, video_embeddings)[0]

            if metric_name == "cosine":
                sorted_idx = np.argsort(-scores)
            else:
                sorted_idx = np.argsort(scores)

            ranked_videos = videos_df.iloc[sorted_idx]["video_id"].tolist()

            # 🔹 TABLE 1 (Ranking)
            for rank in range(5):
                ranking_output.append([
                    queries[i],
                    rank + 1,
                    ranked_videos[rank],
                    float(scores[sorted_idx[rank]])
                ])

            correct_video = ground_truth[i]
            rank_position = ranked_videos.index(correct_video) + 1

            ranks.append(rank_position)

            # 🔹 TABLE 2 (Evaluation)
            evaluation_output.append([
                queries[i],
                correct_video,
                rank_position
            ])

        ranks = np.array(ranks)

        top1 = np.mean(ranks <= 1)
        top3 = np.mean(ranks <= 3)
        top5 = np.mean(ranks <= 5)
        avg_rank = np.mean(ranks)

        # 🔹 TABLE 3 (Comparison)
        all_results.append([
            model_name,
            metric_name,
            round(top1, 3),
            round(top3, 3),
            round(top5, 3),
            round(avg_rank, 2)
        ])


Evaluating all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3101.18it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 3/3 [00:00<00:00, 28.13it/s]



Evaluating all-mpnet-base-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1871.68it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 3/3 [00:00<00:00,  4.82it/s]



Evaluating multi-qa-MiniLM-L6-cos-v1


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4026.93it/s]
BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 3/3 [00:00<00:00, 29.63it/s]


In [8]:
ranking_df = pd.DataFrame(
    ranking_output,
    columns=["Query", "Rank", "Video", "Score"]
)

ranking_df.head(10)

,Query,Rank,Video,Score
0,What is diagram visually?,1,80_RUmj-4B0,0.422286
1,What is diagram visually?,2,gRuI6HzyuPY,0.398598
2,What is diagram visually?,3,la0wPRCAiy0,0.376632
3,What is diagram visually?,4,QxfYZpE9ch8,0.346503
4,What is diagram visually?,5,67Ekk-rM5dE,0.303522
5,Explain performance dns,1,LN-FyvIgwKs,0.587224
6,Explain performance dns,2,Lsd80uR9Shs,0.551143
7,Explain performance dns,3,xv0Be4QfkH0,0.548335
8,Explain performance dns,4,bJ9NnLLMQ78,0.512678
9,Explain performance dns,5,UpZWf1tH6K0,0.494830


In [9]:
eval_df = pd.DataFrame(
    evaluation_output,
    columns=["Query", "Expected Video", "Retrieved Rank"]
)

eval_df.head(10)

,Query,Expected Video,Retrieved Rank
0,What is diagram visually?,67Ekk-rM5dE,5
1,Explain performance dns,xv0Be4QfkH0,3
2,Explain benchmarks dragonfly,j1PkkSddZcE,1
3,How does processing stream work?,mG3xQb_-rV4,1
4,How does sql injection work?,WUBWIVCJLHI,2
5,What is architecture sidecar?,FMxUYhYDiys,1
6,What is apis harder?,qnXTTmPPiz0,1
7,How does advantages fusionaut work?,t3HvCLYnrRY,1
8,Explain legacy microservices,DpuQ3-7e-rY,2
9,What is structured apps?,t-tTh94NI78,8


In [10]:
results_df = pd.DataFrame(
    all_results,
    columns=["Model", "Metric", "Top-1 Recall", "Top-3 Recall", "Top-5 Recall", "Avg Rank"]
)

results_df

,Model,Metric,Top-1 Recall,Top-3 Recall,Top-5 Recall,Avg Rank
0,all-MiniLM-L6-v2,cosine,0.688,0.900,0.950,1.80
1,all-MiniLM-L6-v2,euclidean,0.688,0.900,0.950,1.80
2,all-MiniLM-L6-v2,manhattan,0.675,0.888,0.938,1.89
3,all-mpnet-base-v2,cosine,0.525,0.788,0.850,3.89
4,all-mpnet-base-v2,euclidean,0.525,0.788,0.850,3.89
5,all-mpnet-base-v2,manhattan,0.538,0.775,0.850,4.12
6,multi-qa-MiniLM-L6-cos-v1,cosine,0.612,0.850,0.900,2.78
7,multi-qa-MiniLM-L6-cos-v1,euclidean,0.612,0.850,0.900,2.78
8,multi-qa-MiniLM-L6-cos-v1,manhattan,0.588,0.838,0.900,2.78


In [11]:
ranking_df.to_csv("ranking_output.csv", index=False)
eval_df.to_csv("evaluation_output.csv", index=False)
results_df.to_csv("model_comparison.csv", index=False)